# 🌍 i18n Translator

Dieses Notebook klont das Repository, baut das Frontend und startet die App über einen öffentlichen Cloudflare-Tunnel.

## Voraussetzungen
- **B_API_KEY** als Colab Secret (für KI-Übersetzungsfunktionen)

### B_API_KEY einrichten
1. Links auf das 🔑 Symbol klicken
2. "Neues Secret hinzufügen"
3. Name: `B_API_KEY`, Wert: Ihr API Key
4. "Notebook-Zugriff" aktivieren

> **Hinweis:** Ohne `B_API_KEY` sind alle KI-Funktionen (Auto-Übersetzen, Fehlende füllen, Review) nicht verfügbar – manuelle Bearbeitung funktioniert weiterhin.

## Ablauf
1. Zellen **der Reihe nach** ausführen
2. Am Ende von Zelle 5 erscheint die öffentliche URL
3. Zelle 6 stoppt Backend und Tunnel

## 1️⃣ Repository klonen

In [ ]:
!git clone https://github.com/janschachtschabel/i18nTranslator.git
%cd i18nTranslator
!echo "✅ Repository geklont"


## 2️⃣ B_API_KEY konfigurieren

In [ ]:
import os

try:
    from google.colab import userdata
    key = userdata.get('B_API_KEY')
    if key:
        os.environ['B_API_KEY'] = key
        print('✅ B_API_KEY geladen – KI-Funktionen aktiv')
    else:
        print('⚠️  B_API_KEY ist leer – KI-Funktionen nicht verfügbar')
except Exception as e:
    print(f'⚠️  B_API_KEY nicht gefunden ({e}) – KI-Funktionen nicht verfügbar')


## 3️⃣ Python-Abhängigkeiten installieren

In [ ]:
!pip install -r backend/requirements.txt -q
print('✅ Python-Pakete installiert')


## 4️⃣ Node.js installieren & Frontend bauen

In [ ]:
import subprocess, os

# Node.js 20 installieren
print('📦 Node.js 20 wird installiert...')
subprocess.run('curl -fsSL https://deb.nodesource.com/setup_20.x | bash -', shell=True, capture_output=True)
subprocess.run('apt-get install -y nodejs -qq', shell=True)

import shutil
node_ver = subprocess.check_output(['node', '--version']).decode().strip()
npm_ver  = subprocess.check_output(['npm',  '--version']).decode().strip()
print(f'Node {node_ver}  |  npm {npm_ver}')

# npm-Abhängigkeiten installieren
print('\n📦 npm install...')
subprocess.run('npm install --legacy-peer-deps', shell=True, cwd='/content/i18nTranslator/frontend')

# Production-Build
print('🔨 npm run build...')
result = subprocess.run('npm run build', shell=True, cwd='/content/i18nTranslator/frontend')

if os.path.isdir('/content/i18nTranslator/frontend/dist'):
    print('✅ Frontend erfolgreich gebaut → frontend/dist/')
else:
    print('❌ Build fehlgeschlagen – prüfe Ausgabe oben')


## 5️⃣ App starten & Tunnel öffnen

In [ ]:
import os, sys, subprocess, time, re

# ── Wrapper: FastAPI + statisches Frontend auf Port 8000 ──────────────────────
WRAPPER = '''
import sys, os
sys.path.insert(0, '/content/i18nTranslator/backend')
os.chdir('/content/i18nTranslator/backend')

from main import app
from fastapi.staticfiles import StaticFiles
import uvicorn

# Gebautes Frontend einbinden (nach allen /api-Routen)
app.mount('/', StaticFiles(directory='/content/i18nTranslator/frontend/dist', html=True), name='static')

uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')
'''

with open('/content/colab_server.py', 'w') as f:
    f.write(WRAPPER)

# ── Backend starten ───────────────────────────────────────────────────────────
print('🚀 Backend wird gestartet...')
log = open('/tmp/backend.log', 'w')
backend = subprocess.Popen([sys.executable, '/content/colab_server.py'], stdout=log, stderr=log)
time.sleep(4)

if backend.poll() is not None:
    print('❌ Backend ist abgestürzt. Log:')
    print(open('/tmp/backend.log').read())
    raise SystemExit

print(f'✅ Backend läuft (PID {backend.pid})')

# ── Cloudflared installieren ──────────────────────────────────────────────────
print('📡 Cloudflared wird installiert...')
subprocess.run([
    'wget', '-q',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '-O', '/usr/local/bin/cloudflared'
], check=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

# ── Tunnel starten ────────────────────────────────────────────────────────────
print('🌐 Tunnel wird geöffnet...')
tunnel = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

public_url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = tunnel.stdout.readline().decode('utf-8', errors='ignore').strip()
    if not line:
        time.sleep(0.3)
        continue
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

# PIDs merken (für Zelle 6)
os.environ['_BACKEND_PID'] = str(backend.pid)
os.environ['_TUNNEL_PID']  = str(tunnel.pid)

print()
if public_url:
    print('=' * 60)
    print('✅ i18n Translator ist erreichbar unter:')
    print()
    print(f'   🌐  {public_url}')
    print()
    print('Hinweis: URL ändert sich bei jedem Neustart.')
    print('=' * 60)
else:
    print('❌ Tunnel-URL nicht gefunden.')
    print('Backend-Log (letzte 20 Zeilen):')
    lines = open('/tmp/backend.log').readlines()
    print(''.join(lines[-20:]))


## 6️⃣ App stoppen

In [ ]:
import os, signal

for env_var, label in [('_BACKEND_PID', 'Backend'), ('_TUNNEL_PID', 'Tunnel')]:
    pid = os.environ.get(env_var)
    if pid:
        try:
            os.kill(int(pid), signal.SIGTERM)
            print(f'✅ {label} (PID {pid}) gestoppt')
        except ProcessLookupError:
            print(f'ℹ️  {label} lief nicht mehr')
    else:
        print(f'⚠️  {label}-PID nicht gesetzt – wurde Zelle 5 ausgeführt?')
